# A double-well investigation: from trajectories to an MFPT

**Question:** How long does it take a particle starting in the left well to reach the right well? We investigate the same event by quadrature, a grid generator and independent trajectories. The units are dimensionless. Run cells in order; make a prediction before changing each parameter. Install the project from the repository root as in the README.

**Event:** $x_0=-1$, reflecting boundary $a=-2$, absorbing boundary $b=1$. Potential $F(x)=H(x^2-1)^2$, $\beta=1$, constant $D$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
from stochkin import potential, mfpt_quadrature, grid_generator, grid_mfpt, simulate_fpt

x0, a, b = -1.0, -2.0, 1.0
H, beta, D = 2.0, 1.0, 1.0
F = lambda x: potential(x, H)
xx = np.linspace(a, b, 401)
plt.plot(xx, F(xx))
for pos, name in [(a, "reflecting"), (x0, "start"), (0, "barrier"), (b, "target")]:
    plt.axvline(pos, alpha=.4, label=name)
plt.xlabel("coordinate x"); plt.ylabel("free energy / kBT")
plt.legend(); plt.show()

## 1. Predict and calculate

First predict the effect of doubling $D$ at fixed $F$ on equilibrium and the MFPT. Evaluate the backward integral and the grid result. The grid is a finite-state approximation; refine it and report its relative difference from the integral.

In [ ]:
theory = mfpt_quadrature(x0, a, b, F, D, beta)
print(f"Quadrature MFPT: {theory:.4f}")
for count in (51, 101, 201):
    grid = np.linspace(a, b, count)
    Q = grid_generator(grid, F, D, beta)
    estimate = np.interp(x0, grid, grid_mfpt(Q))
    print(f"nodes={count:3d}  grid={estimate:.4f}  relative difference={estimate/theory-1:+.3%}")
print(f"Quadrature at 2D: {mfpt_quadrature(x0,a,b,F,2*D,beta):.4f}")

## 2. First-passage experiment and survival

We record $Y=\min(\tau_b,T_{\rm obs})$ and a hit flag for each independent trajectory. The sample average of $Y$ estimates the **restricted** mean. Its standard error is the sample standard deviation divided by $\sqrt{N}$. A nonzero censored fraction means the full MFPT is not directly observed; a small fraction alone does not prove that the missing tail is negligible.

In [ ]:
horizon, n_paths, dt = 60.0, 800, 0.004
obs, hit = simulate_fpt(n_traj=n_paths, dt=dt, max_time=horizon,
                        x0=x0, a=a, b=b, D=D, beta=beta,
                        barrier=H, seed=42)
restricted = obs.mean()
se = obs.std(ddof=1)/np.sqrt(n_paths)
print(f"hits: {hit.sum()}/{n_paths}; censored fraction: {(~hit).mean():.4f}")
print(f"restricted mean ± SE: {restricted:.4f} ± {se:.4f}")
tgrid = np.linspace(0, horizon, 300, endpoint=False)
survival = np.array([np.mean(obs > t) for t in tgrid])
plt.step(tgrid, survival, where="post")
plt.xlabel("time"); plt.ylabel("survival probability"); plt.show()
print("Compare the restricted mean to the theory only after assessing the missing tail.")

## 3. Time-step and grid refinement

**Predict:** does reducing $\Delta t$ remove Monte Carlo noise? Run three independent ensembles. Record the restricted mean and SE, hit fraction, and comparison with quadrature. Endpoint-only detection can miss crossings. Differences between runs can be statistical; repeat with additional seeds and increase $N$ before interpreting the trend.

In [ ]:
for dt, seed in [(0.004, 101), (0.002, 102), (0.001, 103)]:
    obs, hit = simulate_fpt(n_traj=800, dt=dt, max_time=60.0,
                            x0=x0, a=a, b=b, D=D, beta=beta,
                            barrier=H, seed=seed)
    se = obs.std(ddof=1)/np.sqrt(len(obs))
    print(f"dt={dt:.3f}, restricted mean={obs.mean():.3f} ± {se:.3f}, censored={(~hit).mean():.3%}")
print(f"Continuum quadrature: {theory:.3f}")

## 4. Change one physical choice at a time

**Task A:** Repeat quadrature for $H=1,2,3$ at fixed $D$, then simulate one of them. What changes in the equilibrium distribution and kinetics? Report the parameter values.

**Task B:** Change the absorbing boundary from $b=1$ to $b=0$ while keeping the reflecting wall and start. Are these the same molecular event? Compare MFPTs.

**Task C:** Verify by quadrature that replacing $D$ by $2D$ divides the MFPT by two. Which inference can or cannot be drawn from an equilibrium histogram alone?

In [ ]:
for H_trial in (1.0, 2.0, 3.0):
    F_trial = lambda x, barrier=H_trial: potential(x, barrier)
    print(f"H={H_trial:.1f}: MFPT={mfpt_quadrature(x0,a,b,F_trial,D,beta):.3f}")
for target in (0.0, 1.0):
    print(f"absorbing boundary={target:.1f}: MFPT={mfpt_quadrature(x0,a,target,F,D,beta):.3f}")

## 5. Explain your result

Write 3–5 sentences including: the event, the initial preparation, the method and uncertainty, the effect of your parameter change, and at least one limitation of extending this result to a molecular collective variable. Remember that $1/\mathrm{MFPT}$ is not automatically a phenomenological rate constant.